# Draws the stratified 120-pair subset (`tab:generative`, `tab:app-generative`)

Defines the 120 category-stratified pairs (`pairs_120/pairs.csv`) used by every 120-pair table: `tab:generative`, `tab:app-generative`, `tab:runtime`, `tab:app-primitives`, `app:ceiling`.

**Needs:** DTD only.

**Maintained runnable path:** `texton_matching.data.build_manifest_120` reproduces this sampler (ported verbatim, see that module's docstring). Kept for provenance.

In [ ]:
# ============================================================
# ENDPOINT-PAIR SUBSET FOR THE GENERATIVE COMPARISON — standalone, Drive only
# Output: DATA_ROOT/pairs_120/   +   pairs_120.zip
#   pairs.csv        k, category_A, category_B, file_A, file_B
#   A/ , B/          0000_A.png / 0000_B.png  (same k = same pair)
#   preview/         0000_AB.png  side-by-side, labelled — for a quick look
#   README.txt
# ============================================================
import os, json, random, zipfile, collections
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
from google.colab import drive
drive.mount('/content/drive')

BASE=os.environ.get('DATA_ROOT', '/content/data')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy; SCALE=f'{BASE}/tm_scale1000'
OUTD=f'{BASE}/pairs_120'; N_PAIRS=120; SEED=2026; RES=256

# ---- load manifest, recover categories
man=json.load(open(f'{SCALE}/manifest.json'))['pairs']
def cat(p,side):
    for key in (f'cat_{side}',f'category_{side}',f'{side}_cat',f'{side}_category'):
        if key in p: return p[key]
    f=str(p.get(f'{side}_file',p.get(side,'')))
    return os.path.basename(os.path.dirname(f)) if '/' in f else 'unknown'
rows=[dict(k=int(p['k']),category_A=cat(p,'A'),category_B=cat(p,'B')) for p in man
      if os.path.exists(f'{SCALE}/A/{int(p["k"]):04d}.png') and os.path.exists(f'{SCALE}/B/{int(p["k"]):04d}.png')]
full=pd.DataFrame(rows)

# ---- stratified sample: same category-of-A proportions as the full 1000, fixed seed
rng=random.Random(SEED); by=collections.defaultdict(list)
for r in rows: by[r['category_A']].append(r['k'])
quota={c:max(1,round(N_PAIRS*len(ks)/len(rows))) for c,ks in by.items()}
chosen=[]
for c,ks in by.items(): chosen+=rng.sample(ks,min(quota[c],len(ks)))
rng.shuffle(chosen); chosen=sorted(chosen[:N_PAIRS])
sub=full[full.k.isin(chosen)].sort_values('k').reset_index(drop=True)

# ---- distribution check: subset vs full, category-of-A shares
cmp=pd.concat([full.category_A.value_counts(normalize=True).rename('full'),
               sub.category_A.value_counts(normalize=True).rename('subset')],axis=1).fillna(0)
print(f'{len(sub)} pairs, {sub.category_A.nunique()}/{full.category_A.nunique()} A-categories represented')
print('max |share diff| across categories:',round(float((cmp.full-cmp.subset).abs().max()),3))

# ---- write files
for d in ['A','B','preview']: os.makedirs(f'{OUTD}/{d}',exist_ok=True)
font=ImageFont.load_default(); out_rows=[]
for k in sub.k:
    A=Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB').resize((RES,RES))
    B=Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB').resize((RES,RES))
    A.save(f'{OUTD}/A/{k:04d}_A.png'); B.save(f'{OUTD}/B/{k:04d}_B.png')
    pv=Image.new('RGB',(2*RES+30,RES+22),'white'); pv.paste(A,(10,20)); pv.paste(B,(RES+20,20))
    d=ImageDraw.Draw(pv); d.text((10,4),f'pair {k:04d}  A',fill='black',font=font); d.text((RES+20,4),'B',fill='black',font=font)
    pv.save(f'{OUTD}/preview/{k:04d}_AB.png')
    r=sub[sub.k==k].iloc[0]
    out_rows.append(dict(k=f'{k:04d}',category_A=r.category_A,category_B=r.category_B,file_A=f'A/{k:04d}_A.png',file_B=f'B/{k:04d}_B.png'))
pd.DataFrame(out_rows).to_csv(f'{OUTD}/pairs.csv',index=False)

open(f'{OUTD}/README.txt','w').write(f"""Texture interpolation — endpoint pairs ({len(sub)} pairs)
Stratified subset of our 1000-pair DTD evaluation set (same category proportions, seed {SEED}, {RES}x{RES} px).

Layout
  pairs.csv    one row per pair: k, category_A, category_B, file_A, file_B
  A/, B/       the two source textures of each pair; the same 4-digit k = the same pair
  preview/     A|B side by side per pair, for browsing only

Task
  For each pair, produce the midpoint texture between A and B (eta = 0.5): a single coherent texture
  that lies halfway between the two materials — not a cross-fade or overlay of both.
  Please return one image per pair named  {{k}}_mid.png  (e.g. 0042_mid.png), same k as the inputs,
  at 256x256 if possible. Keep the k naming exactly — it is how we match outputs to pairs for scoring.
""")

zp=f'{BASE}/pairs_120.zip'
with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as z:
    for root,_,fs in os.walk(OUTD):
        for f in fs: z.write(os.path.join(root,f),os.path.relpath(os.path.join(root,f),OUTD))
mb=os.path.getsize(zp)/1e6
print(f'{zp}: {mb:.0f} MB ->','ok for git' if mb<95 else 'share the Drive folder instead')

In [ ]:
print(man[0])

In [ ]:
# ============================================================
# ENDPOINT-PAIR SUBSET FOR THE GENERATIVE COMPARISON — v2 (category detection fixed) — standalone, Drive only
# ============================================================
import os, json, random, zipfile, collections, shutil
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
from google.colab import drive
drive.mount('/content/drive')

BASE=os.environ.get('DATA_ROOT', '/content/data')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy; SCALE=f'{BASE}/tm_scale1000'
OUTD=f'{BASE}/pairs_120'; N_PAIRS=120; SEED=2026; RES=256
if os.path.isdir(OUTD): shutil.rmtree(OUTD)          # clean the earlier unstratified run

man=json.load(open(f'{SCALE}/manifest.json'))['pairs']
print('manifest keys:', list(man[0].keys())); print('example:', man[0])

def cat(p,side): return p[f'c{side}']

rows=[dict(k=int(p['k']),category_A=cat(p,'A'),category_B=cat(p,'B')) for p in man
      if os.path.exists(f'{SCALE}/A/{int(p["k"]):04d}.png') and os.path.exists(f'{SCALE}/B/{int(p["k"]):04d}.png')]
full=pd.DataFrame(rows)
assert full.category_A.nunique()>1, "category detection still failed — paste the 'example:' line above"

rng=random.Random(SEED); by=collections.defaultdict(list)
for r in rows: by[r['category_A']].append(r['k'])
quota={c:max(1,round(N_PAIRS*len(ks)/len(rows))) for c,ks in by.items()}
chosen=[]
for c,ks in by.items(): chosen+=rng.sample(ks,min(quota[c],len(ks)))
rng.shuffle(chosen); chosen=sorted(chosen[:N_PAIRS])
sub=full[full.k.isin(chosen)].sort_values('k').reset_index(drop=True)
cmp=pd.concat([full.category_A.value_counts(normalize=True).rename('full'),
               sub.category_A.value_counts(normalize=True).rename('subset')],axis=1).fillna(0)
print(f'{len(sub)} pairs, {sub.category_A.nunique()}/{full.category_A.nunique()} A-categories represented')
print('max |share diff| across categories:',round(float((cmp.full-cmp.subset).abs().max()),3))

for d in ['A','B','preview']: os.makedirs(f'{OUTD}/{d}',exist_ok=True)
font=ImageFont.load_default(); out_rows=[]
for k in sub.k:
    A=Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB').resize((RES,RES))
    B=Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB').resize((RES,RES))
    A.save(f'{OUTD}/A/{k:04d}_A.png'); B.save(f'{OUTD}/B/{k:04d}_B.png')
    pv=Image.new('RGB',(2*RES+30,RES+22),'white'); pv.paste(A,(10,20)); pv.paste(B,(RES+20,20))
    d=ImageDraw.Draw(pv); d.text((10,4),f'pair {k:04d}  A',fill='black',font=font); d.text((RES+20,4),'B',fill='black',font=font)
    pv.save(f'{OUTD}/preview/{k:04d}_AB.png')
    r=sub[sub.k==k].iloc[0]
    out_rows.append(dict(k=f'{k:04d}',category_A=r.category_A,category_B=r.category_B,file_A=f'A/{k:04d}_A.png',file_B=f'B/{k:04d}_B.png'))
pd.DataFrame(out_rows).to_csv(f'{OUTD}/pairs.csv',index=False)

open(f'{OUTD}/README.txt','w').write(f"""Texture interpolation — endpoint pairs ({len(sub)} pairs)
Stratified subset of our 1000-pair DTD evaluation set (same category proportions, seed {SEED}, {RES}x{RES} px).

Layout
  pairs.csv    one row per pair: k, category_A, category_B, file_A, file_B
  A/, B/       the two source textures of each pair; the same 4-digit k = the same pair
  preview/     A|B side by side per pair, for browsing only

Task
  For each pair, produce the midpoint texture between A and B (eta = 0.5): a single coherent texture
  that lies halfway between the two materials — not a cross-fade or overlay of both.
  Please return one image per pair named  {{k}}_mid.png  (e.g. 0042_mid.png), same k as the inputs,
  at 256x256 if possible. Keep the k naming exactly — it is how we match outputs to pairs for scoring.
""")

zp=f'{BASE}/pairs_120.zip'
if os.path.exists(zp): os.remove(zp)
with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as z:
    for root,_,fs in os.walk(OUTD):
        for f in fs: z.write(os.path.join(root,f),os.path.relpath(os.path.join(root,f),OUTD))
print(f'{zp}: {os.path.getsize(zp)/1e6:.0f} MB')